In [0]:
%pip install geopandas leafmap mapclassify --quiet

In [0]:
from pyspark.sql import functions as F

# Read municipal population data with geometries
df_all = spark.table("geospatial.spain_population_analysis.padron_municipios_geo")

# Find the latest period and filter to it (table contains multiple years)
latest_period = df_all.select(F.max("periodo")).collect()[0][0]
print(f"Latest period: {latest_period}")

df = df_all.filter(F.col("periodo") == latest_period)

# Sample to verify structure
print(f"Total municipalities: {df.count():,}")
total_pop = df.select(F.sum('poblacion')).collect()[0][0]
if total_pop:
    print(f"Total population: {total_pop:,}")

display(df.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import explode, size

# Convert to H3 resolution 9 using Databricks H3 functions
# Resolution 9 provides detailed granularity (~4.8M cells nationally)
# Area-weighted population distribution: each cell gets population proportional to cell count

print("Converting geometries to H3 cells at resolution 9...")

# Simplify geometry to reduce vertex count before H3 polyfill
# This reduces memory pressure during the polyfill operation
df_with_h3 = df.selectExpr(
    "codigo_municipio",
    "poblacion",
    "st_simplify(geometry, 0.001) as simplified_geom"
).selectExpr(
    "codigo_municipio",
    "poblacion",
    "st_aswkt(simplified_geom) as geom_wkt"
).selectExpr(
    "codigo_municipio",
    "poblacion",
    "h3_polyfillash3string(geom_wkt, 9) as h3_cells"
).withColumn(
    "num_cells",
    size("h3_cells")
)

print("Exploding H3 cells and distributing population...")

# Explode and distribute population evenly across cells
df_h3_exploded = df_with_h3.select(
    "codigo_municipio",
    "poblacion",
    "num_cells",
    explode("h3_cells").alias("h3_cell")
).withColumn(
    "poblacion_cell",
    F.col("poblacion") / F.col("num_cells")
)

print("Aggregating population by H3 cell...")

# Group by H3 cell to aggregate population
# NOT materializing geometries at res-9 (~4.8M cells) - too expensive
# Will create geometries only at res-7 and res-6 for visualization
df_h3_res9 = df_h3_exploded.groupBy("h3_cell").agg(
    F.sum("poblacion_cell").alias("poblacion")
).withColumn(
    "h3_resolution",
    F.lit(9)
).select(
    "h3_cell",
    "h3_resolution",
    "poblacion"
)

print(f"\nTotal H3 cells (res-9): {df_h3_res9.count():,}")
print(f"Total population in H3: {df_h3_res9.select(F.sum('poblacion')).collect()[0][0]:,.0f}")

display(df_h3_res9.limit(10))

In [0]:
from pyspark.sql import functions as F

# Roll up res-9 cells to resolution 6 with geometries for visualization
# Only materializing geometries at res-6 (~13K cells) for national overview
# Not persisting - going directly to plotting

print("Rolling up res-9 to res-6 for visualization...")

df_h3_res6 = df_h3_res9.selectExpr(
    "h3_toparent(h3_cell, 6) as h3_cell",
    "poblacion"
).groupBy("h3_cell").agg(
    F.sum("poblacion").alias("poblacion")
).withColumn(
    "h3_resolution",
    F.lit(6)
).selectExpr(
    "h3_cell",
    "h3_resolution",
    "poblacion",
    "h3_boundaryaswkb(h3_cell) as geometry"
)

print(f"Total H3 cells (res-6): {df_h3_res6.count():,}")
print(f"Total population (res-6): {df_h3_res6.select(F.sum('poblacion')).collect()[0][0]:,.0f}")

display(df_h3_res6.limit(10))

In [0]:
import geopandas as gpd
from shapely import wkb

# Convert H3 res-6 to GeoDataFrame for leafmap visualization
# Using res-6 (~13K cells) for national overview

pdf_h3 = df_h3_res6.toPandas()

# Convert WKB geometry to shapely geometries
pdf_h3['geometry'] = pdf_h3['geometry'].apply(lambda x: wkb.loads(bytes(x)))

# Create GeoDataFrame
gdf_h3 = gpd.GeoDataFrame(pdf_h3, geometry='geometry', crs='EPSG:4326')

# Calculate density (population per km²)
gdf_h3['area_km2'] = gdf_h3.geometry.to_crs('EPSG:3035').area / 1_000_000
gdf_h3['densidad'] = gdf_h3['poblacion'] / gdf_h3['area_km2']

print(f"GeoDataFrame shape: {gdf_h3.shape}")
print(f"\nPopulation statistics:")
print(gdf_h3['poblacion'].describe())
print(f"\nDensity statistics (per km²):")
print(gdf_h3['densidad'].describe())

gdf_h3.head()

In [0]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Filter out cells with very low population for cleaner visualization
gdf_h3_filtered = gdf_h3[gdf_h3['poblacion'] > 10].copy()

# Create figure and axis
fig, ax = plt.subplots(1, 1, figsize=(16, 12))

# Plot H3 cells with log scale for better visualization of density distribution
gdf_h3_filtered.plot(
    column='densidad',
    ax=ax,
    cmap='YlOrRd',
    edgecolor='none',
    legend=True,
    norm=LogNorm(vmin=gdf_h3_filtered['densidad'].min(), vmax=gdf_h3_filtered['densidad'].max()),
    legend_kwds={'label': 'Population Density (per km²)', 'shrink': 0.7}
)

# Set title and labels
ax.set_title('Spain Population Density - H3 Resolution 6', fontsize=16, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

# Remove axis spines for cleaner look
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

# Add grid
ax.grid(True, alpha=0.3, linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

print(f"\nVisualized {len(gdf_h3_filtered):,} H3 cells (res-6)")
print(f"Density range: {gdf_h3_filtered['densidad'].min():.2f} - {gdf_h3_filtered['densidad'].max():.2f} per km²")